In [1]:
import math
import re
import json
from pathlib import Path
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import plotly.graph_objects as go
import plotly.express as px
from tqdm.notebook import tqdm   # progress bar

pd.set_option("display.max_columns", None)
plt.style.use("dark_background")

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("C:/xcas-ga-comms-assistant")
ADSB_CLEAN    = PROJECT_ROOT / "data/interim/adsb_2020-10-22_clean.csv"
WEATHER_PATH  = PROJECT_ROOT / "tartan_data/weather/BTP.csv"
OUTPUT_DIR    = PROJECT_ROOT / "data/interim"

# ── Airport constants (KBTP — Butler County Regional, PA) ────────────────────
KBTP = dict(
    icao      = "KBTP",
    name      = "Butler",
    lat       = 40.7769,
    lon       = -79.9697,
    elevation = 1248,            # ft MSL (average of 1227/1243 threshold elevations)

    # ── Official FAA runway data (verified via FAA Airport Diagram) ───────────
    # Source: FAA NASR — True Alignment 072° / 252°
    # KBTP has ONE runway: 08/26 (named by magnetic heading, true is 072/252)
    # Length: 4801 ft × 100 ft, Asphalt
    # Both ends: Left traffic pattern
    runways = [
        dict(id="08", hdg=72,  length=4801, reciprocal="26",
             pattern="left", ils=False),
        dict(id="26", hdg=252, length=4801, reciprocal="08",
             pattern="left", ils=True),   # Runway 26 has ILS + PAPI
    ],

    ctaf_freq = "123.05",
    pattern   = "left",
)

print("✅ KBTP airport constants updated with verified FAA data")
print(f"   Runway 08: true heading {KBTP['runways'][0]['hdg']}°, "
      f"{KBTP['runways'][0]['length']} ft")
print(f"   Runway 26: true heading {KBTP['runways'][1]['hdg']}°, "
      f"{KBTP['runways'][1]['length']} ft, ILS approach")
print(f"   Single runway pair only — removed phantom 18/36")
# Load clean ADS-B
df_adsb = pd.read_csv(ADSB_CLEAN, parse_dates=["timestamp"])
print(f"✅ Loaded ADS-B: {len(df_adsb):,} rows, {df_adsb['Tail'].nunique()} aircraft")
print(f"   Time: {df_adsb['timestamp'].min().time()} → {df_adsb['timestamp'].max().time()}")

✅ KBTP airport constants updated with verified FAA data
   Runway 08: true heading 72°, 4801 ft
   Runway 26: true heading 252°, 4801 ft, ILS approach
   Single runway pair only — removed phantom 18/36
✅ Loaded ADS-B: 219,622 rows, 257 aircraft
   Time: 08:20:08.935000 → 23:59:59.194000


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# METAR PARSING & ACTIVE RUNWAY DETERMINATION
#
# 📚 Concept — METAR:
#   METAR is the international standard weather report format used at airports.
#   It's a compact string encoding: temperature, dewpoint, wind, visibility,
#   cloud cover, altimeter setting. Example:
#   "METAR KBTP 221455Z 24012KT 10SM CLR 15/04 A3012"
#   Broken down:
#     KBTP     = station
#     221455Z  = day 22, time 1455 UTC (Z = Zulu = UTC)
#     24012KT  = wind FROM 240° at 12 knots
#     10SM     = visibility 10 statute miles
#     CLR      = clear skies
#     15/04    = temp 15°C / dewpoint 4°C
#     A3012    = altimeter 30.12 inHg
#
# 📚 Runway selection rule:
#   Pilots land INTO the wind. The active runway heading should be within
#   ±90° of the wind direction. Specifically: choose the runway whose
#   heading gives the smallest crosswind component.
#   Rule of thumb: "Land on the runway closest to the wind direction."
# ═══════════════════════════════════════════════════════════════════════════════

def parse_metar_wind(metar_string: str) -> Optional[tuple[float, float]]:
    """
    Extract wind direction (degrees) and speed (knots) from a METAR string.
    Returns (direction_deg, speed_kts) or None if not parseable.
    
    Handles:
        "27010KT"   → (270, 10)
        "VRB05KT"   → (None, 5)  variable direction
        "00000KT"   → (0, 0)     calm
    """
    # Pattern: 3-digit direction + 2-3 digit speed + "KT"
    m = re.search(r'(\d{3})(\d{2,3})KT', str(metar_string))
    if m:
        direction = float(m.group(1))
        speed     = float(m.group(2))
        return direction, speed
    
    # Variable wind
    m_vrb = re.search(r'VRB(\d{2,3})KT', str(metar_string))
    if m_vrb:
        return None, float(m_vrb.group(1))   # None = variable direction
    
    return None


def select_active_runway(wind_dir_deg: float, airport: dict) -> dict:
    """
    Given wind direction, select the runway with the best headwind component.
    
    📚 Headwind component = speed × cos(angle_between_wind_and_runway)
    We pick the runway where this cos() is maximised (angle closest to 0°).
    
    Example:
        Wind from 240°, runway 26 heading = 258°
        Angle difference = |240 - 258| = 18°  → cos(18°) = 0.95 → strong headwind ✅
        Runway 08 heading = 078°
        Angle difference = |240 - 078| = 162° → cos(162°) = -0.95 → tailwind ✗
    """
    best_runway = None
    best_score  = -999
    
    for rwy in airport["runways"]:
        hdg = rwy["hdg"]
        # Angle between wind direction and runway heading
        # Wind FROM 240° means aircraft flying INTO wind points toward 240°
        # Runway 26 heading = 258° → aircraft lands heading 258°, into 240° wind ✅
        diff = abs(wind_dir_deg - hdg)
        # Normalise to 0-180 range
        if diff > 180:
            diff = 360 - diff
        # Headwind score: 0° diff = perfect headwind (score=1), 180° = tailwind (score=-1)
        headwind_score = math.cos(math.radians(diff))
        if headwind_score > best_score:
            best_score  = headwind_score
            best_runway = rwy
    
    return best_runway


# ── Load and parse BTP.csv ─────────────────────────────────────────────────────
df_wx = pd.read_csv(WEATHER_PATH)
print("Weather columns:", list(df_wx.columns))
print(f"Shape: {df_wx.shape}")
display(df_wx.head(3))

Weather columns: ['station', 'valid', 'lon', 'lat', 'elevation', 'tmpf', 'dwpf', 'relh', 'drct', 'sknt', 'p01i', 'alti', 'mslp', 'vsby', 'gust', 'skyc1', 'skyc2', 'skyc3', 'skyc4', 'skyl1', 'skyl2', 'skyl3', 'skyl4', 'wxcodes', 'ice_accretion_1hr', 'ice_accretion_3hr', 'ice_accretion_6hr', 'peak_wind_gust', 'peak_wind_drct', 'peak_wind_time', 'feel', 'metar', 'snowdepth']
Shape: (35964, 33)


C:\Users\vijay\AppData\Local\Temp\ipykernel_5184\2169559332.py:85: DtypeWarning: Columns (0: tmpf, 1: dwpf, 2: relh, 3: vsby) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wx = pd.read_csv(WEATHER_PATH)


,station,valid,lon,lat,elevation,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,vsby,gust,skyc1,skyc2,skyc3,skyc4,skyl1,skyl2,skyl3,skyl4,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth
0,BTP,2020-01-01 00:05,-79.9497,40.7769,380.0,30.20,28.40,92.92,260.00,8.00,T,29.72,M,2.50,M,BKN,BKN,OVC,M,800.00,1300.00,1700.00,M,-SN BR,M,M,M,M,M,M,21.90,KBTP 010005Z AUTO 26008KT 2 1/2SM -SN BR BKN00...,M
1,BTP,2020-01-01 00:17,-79.9497,40.7769,380.0,30.20,28.40,92.92,250.00,8.00,0.02,29.72,M,1.50,M,BKN,OVC,M,M,600.00,1100.00,M,M,-SN BR,M,M,M,M,M,M,21.90,KBTP 010017Z AUTO 25008KT 1 1/2SM -SN BR BKN00...,M
2,BTP,2020-01-01 00:31,-79.9497,40.7769,380.0,30.20,28.40,92.92,240.00,10.00,0.02,29.72,M,3.00,M,BKN,OVC,M,M,600.00,1100.00,M,M,-SN BR,M,M,M,M,M,M,20.71,KBTP 010031Z AUTO 24010KT 3SM -SN BR BKN006 OV...,M


In [4]:
# ── Filter weather to our target date ────────────────────────────────────────
# BTP.csv has METAR data — find the column that contains the raw METAR string
# Common column names: 'metar', 'rawOb', 'raw_ob', 'presentwx'
# First let's see what we actually have:
print("Weather columns and sample values:")
for col in df_wx.columns:
    print(f"  {col:20s}: {df_wx[col].iloc[0]}")

Weather columns and sample values:
  station             : BTP
  valid               : 2020-01-01 00:05
  lon                 : -79.9497
  lat                 : 40.7769
  elevation           : 380.0
  tmpf                : 30.20
  dwpf                : 28.40
  relh                : 92.92
  drct                : 260.00
  sknt                : 8.00
  p01i                : T
  alti                : 29.72
  mslp                : M
  vsby                : 2.50
  gust                : M
  skyc1               : BKN
  skyc2               : BKN
  skyc3               : OVC
  skyc4               : M
  skyl1               : 800.00
  skyl2               : 1300.00
  skyl3               : 1700.00
  skyl4               : M
  wxcodes             : -SN BR
  ice_accretion_1hr   : M
  ice_accretion_3hr   : M
  ice_accretion_6hr   : M
  peak_wind_gust      : M
  peak_wind_drct      : M
  peak_wind_time      : M
  feel                : 21.90
  metar               : KBTP 010005Z AUTO 26008KT 2 1/2SM -SN BR B

In [6]:
# ── After inspecting columns, identify date and metar columns ────────────────
# Adjust column names below based on what you see printed above

# Common BTP.csv structure from Iowa State METAR archive:
# station, valid, tmpf, dwpf, relh, drct, sknt, p01i, alti, mslp, vsby, ...

# Find the datetime column (usually 'valid' in Iowa State format)
date_col   = "valid"      # ← adjust if different
wind_dir_col = "drct"     # wind direction in degrees
wind_spd_col = "sknt"     # wind speed in knots

In [7]:
# ── Filter weather to our target date ────────────────────────────────────────
df_wx["dt"] = pd.to_datetime(df_wx[date_col], errors="coerce")
df_wx_day = df_wx[df_wx["dt"].dt.date == pd.Timestamp("2020-10-22").date()].copy()
print(f"METAR records for 2020-10-22: {len(df_wx_day)}")

# ── Force numeric conversion BEFORE any comparison ───────────────────────────
# errors="coerce" turns anything unparseable ("VRB", "M", "") into NaN
# instead of crashing — then dropna() removes those rows cleanly
df_wx_day[wind_dir_col] = pd.to_numeric(df_wx_day[wind_dir_col], errors="coerce")
df_wx_day[wind_spd_col] = pd.to_numeric(df_wx_day[wind_spd_col], errors="coerce")

print(f"\nWind speed dtype  : {df_wx_day[wind_spd_col].dtype}")
print(f"Wind dir  dtype  : {df_wx_day[wind_dir_col].dtype}")
print(f"Speed range      : {df_wx_day[wind_spd_col].min():.0f} – {df_wx_day[wind_spd_col].max():.0f} kts")
print(f"Non-null speed   : {df_wx_day[wind_spd_col].notna().sum()} / {len(df_wx_day)} rows")

display(df_wx_day[[date_col, wind_dir_col, wind_spd_col]].head(10))

METAR records for 2020-10-22: 44

Wind speed dtype  : float64
Wind dir  dtype  : float64
Speed range      : 0 – 9 kts
Non-null speed   : 44 / 44 rows


,valid,drct,sknt
9624,2020-10-22 00:56,0.0,0.0
9625,2020-10-22 01:22,0.0,0.0
9626,2020-10-22 01:56,0.0,0.0
9627,2020-10-22 02:56,0.0,0.0
9628,2020-10-22 03:56,0.0,0.0
9629,2020-10-22 04:47,0.0,0.0
9630,2020-10-22 04:56,0.0,0.0
9631,2020-10-22 04:59,0.0,0.0
9632,2020-10-22 05:07,0.0,0.0
9633,2020-10-22 05:31,0.0,0.0


In [ ]:
# ── Determine predominant wind and active runway for the day ─────────────────
# Filter to rows where wind speed is meaningful (≥ 3 kts) and both cols are valid
valid_wind = df_wx_day[
    df_wx_day[wind_spd_col].notna() &
    df_wx_day[wind_dir_col].notna() &
    (df_wx_day[wind_spd_col] >= 3)
].copy()

print(f"Valid wind observations (≥3 kts): {len(valid_wind)} of {len(df_wx_day)}")

if len(valid_wind) > 0:
    # Circular mean — correct way to average angles
    # 📚 Normal mean of [350°, 10°] = 180° (WRONG — that's South)
    #    Circular mean converts to unit vectors first, averages, converts back
    angles_rad = np.deg2rad(valid_wind[wind_dir_col].values)
    mean_sin = np.sin(angles_rad).mean()
    mean_cos = np.cos(angles_rad).mean()
    circular_mean_wind = float(np.rad2deg(np.arctan2(mean_sin, mean_cos)) % 360)
    median_speed = float(valid_wind[wind_spd_col].median())
    print(f"Wind direction (circular mean): {circular_mean_wind:.0f}°")
    print(f"Wind speed     (median)       : {median_speed:.0f} kts")
else:
    circular_mean_wind = 270.0
    median_speed = 5.0
    print("⚠  No valid wind data — defaulting to 270° (westerly)")

# In notebook 02, replace the active runway selection cell with this:

def select_active_runway_v2(wind_dir_deg, wind_speed_kts, airport, 
                             default_runway_id="26"):
    """
    Select active runway considering:
    - If wind >= 6 kts: use headwind calculation (standard)
    - If wind < 6 kts (calm): use default preferred runway
    
    📚 On calm days pilots choose runway based on:
      - Preferred instrument approach (ILS on 26 at KBTP)
      - Intended departure direction
      - Local convention
    KBTP default is 26 (ILS approach, preferred by most pilots)
    """
    if wind_speed_kts >= 6:
        return select_active_runway(wind_dir_deg, airport)
    else:
        # Calm wind — return preferred runway
        for rwy in airport["runways"]:
            if rwy["id"] == default_runway_id:
                print(f"  Calm wind ({wind_speed_kts:.0f} kts) "
                      f"→ using preferred runway {default_runway_id}")
                return rwy
        return airport["runways"][0]

# On Oct 22 2020: wind=4kts → calm → use Runway 26
active_runway = select_active_runway_v2(
    circular_mean_wind, median_speed, KBTP, default_runway_id="26"
)
print(f"Active runway (v2): {active_runway['id']} "
      f"(hdg {active_runway['hdg']}°)")

print(f"\n✅ Active runway for 2020-10-22: {active_runway['id']}")
print(f"   Runway heading : {active_runway['hdg']}°")
print(f"   Runway length  : {active_runway['length']:,} ft")
print(f"   Wind from      : {circular_mean_wind:.0f}°")
print(f"   (Pilot lands heading {active_runway['hdg']}° into {circular_mean_wind:.0f}° wind)")

Valid wind observations (≥3 kts): 20 of 44
Wind direction (circular mean): 154°
Wind speed     (median)       : 4 kts

✅ Active runway for 2020-10-22: 08
   Runway heading : 72°
   Runway length  : 4,801 ft
   Wind from      : 154°
   (Pilot lands heading 72° into 154° wind)


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# AVIATION GEOMETRY TOOLKIT
#
# 📚 Concept — Why not just use lat/lon differences?
#   The Earth is a sphere (approximately). Treating lat/lon as flat Cartesian
#   coordinates introduces errors that grow with distance. At 40°N latitude,
#   1 degree of longitude ≠ 1 degree of latitude in distance.
#   We use the Haversine formula for accurate great-circle distances.
# ═══════════════════════════════════════════════════════════════════════════════

def bearing_to_compass(bearing_deg: float) -> str:
    """
    Convert a numeric bearing to a compass direction string.
    Used in callout generation: bearing 015° → "North"
    
    📚 Compass quadrants for radio calls:
       N = 315–360 and 0–45
       NE = 45–90, E = 90–135... etc.
    """
    bearing_deg = bearing_deg % 360
    directions = [
        (22.5,  "North"),
        (67.5,  "Northeast"),
        (112.5, "East"),
        (157.5, "Southeast"),
        (202.5, "South"),
        (247.5, "Southwest"),
        (292.5, "West"),
        (337.5, "Northwest"),
        (360.0, "North"),
    ]
    for threshold, name in directions:
        if bearing_deg <= threshold:
            return name
    return "North"


def bearing_from_airport(aircraft_lat: float, aircraft_lon: float,
                          airport_lat: float,  airport_lon: float) -> float:
    """
    Calculate bearing FROM airport TO aircraft.
    This tells us which direction the aircraft is approaching FROM.
    
    Returns bearing in degrees (0=North, 90=East, 180=South, 270=West).
    
    📚 Haversine-based bearing formula:
       Uses atan2 of sin/cos components — standard spherical navigation.
    """
    lat1 = math.radians(airport_lat)
    lat2 = math.radians(aircraft_lat)
    dlon = math.radians(aircraft_lon - airport_lon)
    
    x = math.sin(dlon) * math.cos(lat2)
    y = (math.cos(lat1) * math.sin(lat2) -
         math.sin(lat1) * math.cos(lat2) * math.cos(dlon))
    
    bearing = math.degrees(math.atan2(x, y))
    return bearing % 360   # normalise to 0–360


def km_to_nm(km: float) -> float:
    """Convert kilometres to nautical miles. 1 NM = 1.852 km exactly."""
    return km / 1.852


def classify_pattern_leg(aircraft_lat: float, aircraft_lon: float,
                          aircraft_hdg: float, aircraft_alt: float,
                          airport_lat: float, airport_lon: float,
                          active_runway: dict, airport_elevation: float,
                          pattern_altitude_ft: float = 1000.0) -> Optional[str]:
    """
    Determine if an aircraft is on a specific traffic pattern leg.
    
    📚 Traffic pattern geometry:
       Standard left-hand pattern legs relative to runway:
       
       UPWIND  → heading matches runway heading ± 30°, climbing, close
       CROSSWIND → heading ≈ runway + 90° (left turn)
       DOWNWIND → heading ≈ runway + 180° (opposite direction), ~1NM abeam
       BASE     → heading ≈ runway + 270° (left turn toward runway)
       FINAL    → heading matches runway heading ± 30°, descending, aligned
       
    Returns leg name or None if not in pattern.
    """
    rwy_hdg = active_runway["hdg"]
    agl = aircraft_alt - airport_elevation  # altitude above ground level
    
    # Pattern altitude is typically 800-1000 ft AGL for GA
    # Aircraft must be near pattern altitude to be in pattern
    if agl < 200 or agl > 2000:
        return None
    
    # Heading alignment checks (with ±35° tolerance for GA)
    hdg_diff_final   = abs(((aircraft_hdg - rwy_hdg) + 180) % 360 - 180)
    hdg_diff_downwind= abs(((aircraft_hdg - (rwy_hdg + 180)) + 180) % 360 - 180)
    hdg_diff_base    = abs(((aircraft_hdg - (rwy_hdg + 270)) + 180) % 360 - 180)  # left
    hdg_diff_crosswind=abs(((aircraft_hdg - (rwy_hdg + 90)) + 180) % 360 - 180)   # left
    
    range_nm = km_to_nm(
        # Recalculate distance using haversine for accuracy
        2 * 6371 * math.asin(math.sqrt(
            math.sin(math.radians((aircraft_lat - airport_lat)/2))**2 +
            math.cos(math.radians(airport_lat)) *
            math.cos(math.radians(aircraft_lat)) *
            math.sin(math.radians((aircraft_lon - airport_lon)/2))**2
        ))
    )
    
    if range_nm > 5:
        return None   # too far to be in pattern
    
    TOLERANCE = 35   # degrees heading tolerance
    
    if hdg_diff_final <= TOLERANCE and range_nm <= 3:
        return "FINAL"
    elif hdg_diff_base <= TOLERANCE and range_nm <= 3:
        return "BASE"
    elif hdg_diff_downwind <= TOLERANCE and 0.5 <= range_nm <= 2.5:
        return "DOWNWIND"
    elif hdg_diff_crosswind <= TOLERANCE and range_nm <= 2:
        return "CROSSWIND"
    
    return None


# Quick test
test_bearing = bearing_from_airport(40.9, -79.97, KBTP["lat"], KBTP["lon"])
print(f"Test bearing (aircraft N of airport): {test_bearing:.1f}° → {bearing_to_compass(test_bearing)}")
print(f"5 km in NM: {km_to_nm(5):.2f} NM")
print(f"10 NM in km: {10 * 1.852:.2f} km")

Test bearing (aircraft N of airport): 359.9° → North
5 km in NM: 2.70 NM
10 NM in km: 18.52 km


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# RUNWAY SELECTION DIAGNOSTIC
# Always verify the algorithm's choice makes physical sense
# ═══════════════════════════════════════════════════════════════════════════════

print("RUNWAY SELECTION DIAGNOSTIC")
print("="*55)
print(f"\nWind: FROM {circular_mean_wind:.0f}° at {median_speed:.0f} kts")
print(f"(Wind from 154° = blowing from SE toward NW)\n")

for rwy in KBTP["runways"]:
    diff = abs(circular_mean_wind - rwy["hdg"])
    if diff > 180:
        diff = 360 - diff
    headwind  = median_speed * math.cos(math.radians(diff))
    crosswind = median_speed * math.sin(math.radians(diff))
    print(f"  Runway {rwy['id']} (hdg {rwy['hdg']}°):")
    print(f"    Angle to wind     : {diff:.0f}°")
    print(f"    Headwind component: {headwind:+.2f} kts "
          f"{'← HEADWIND ✅' if headwind > 0 else '← TAILWIND ✗'}")
    print(f"    Crosswind component:{abs(crosswind):.2f} kts")
    print()

print(f"Algorithm selected: Runway {active_runway['id']}")
print(f"\nVerdict:")
if median_speed < 6:
    print(f"  ⚠  Wind is VERY LIGHT ({median_speed:.0f} kts) and nearly full crosswind.")
    print(f"     Selection is mathematically correct but operationally marginal.")
    print(f"     Real pilots at KBTP with calm winds likely use Runway 26 by default")
    print(f"     (ILS approach, preferred instrument approach end).")
    print(f"\n  → For prototype: we'll KEEP Runway 08 (wind-preferred)")
    print(f"    but flag this as an assumption in the README.")
    print(f"    Production system would cross-check AWOS/ATIS broadcast.")
else:
    print(f"  ✅ Wind is strong enough to be meaningful — selection is reliable.")

# ── Check: how many hours had wind favoring each runway? ─────────────────────
print(f"\nHour-by-hour runway preference on 2020-10-22:")
df_wx_day["rwy_preferred"] = df_wx_day[wind_dir_col].apply(
    lambda d: select_active_runway(float(d), KBTP)["id"]
              if pd.notna(d) else "unknown"
)
print(df_wx_day.groupby("rwy_preferred").size().to_string())

RUNWAY SELECTION DIAGNOSTIC

Wind: FROM 154° at 4 kts
(Wind from 154° = blowing from SE toward NW)

  Runway 08 (hdg 72°):
    Angle to wind     : 82°
    Headwind component: +0.56 kts ← HEADWIND ✅
    Crosswind component:3.96 kts

  Runway 26 (hdg 252°):
    Angle to wind     : 98°
    Headwind component: -0.56 kts ← TAILWIND ✗
    Crosswind component:3.96 kts

Algorithm selected: Runway 08

Verdict:
  ⚠  Wind is VERY LIGHT (4 kts) and nearly full crosswind.
     Selection is mathematically correct but operationally marginal.
     Real pilots at KBTP with calm winds likely use Runway 26 by default
     (ILS approach, preferred instrument approach end).

  → For prototype: we'll KEEP Runway 08 (wind-preferred)
    but flag this as an assumption in the README.
    Production system would cross-check AWOS/ATIS broadcast.

Hour-by-hour runway preference on 2020-10-22:
rwy_preferred
08         36
26          7
unknown     1


In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRACK-LEVEL FEATURE ENGINEERING
#
# We compute features that require looking at CONSECUTIVE pings per aircraft:
#   - range_delta_nm  : change in range from last ping (negative = approaching)
#   - alt_delta_ft    : change in altitude from last ping (negative = descending)
#   - is_approaching  : boolean, range decreasing over last N pings
#   - is_descending   : boolean, altitude decreasing over last N pings
#   - agl             : altitude above ground level
#
# 📚 Concept — .groupby().shift():
#   groupby("Tail") splits the dataframe into one group per aircraft.
#   .shift(1) gives you the PREVIOUS row's value within each group.
#   So row[i]["prev_range"] = row[i-1]["range_nm"] for the same aircraft.
#   This is how you compute differences between consecutive observations
#   without writing a loop — it's vectorised and fast.
# ═══════════════════════════════════════════════════════════════════════════════

print("Computing track-level features...")

# Sort by aircraft and time — CRITICAL before shift()
df_adsb = df_adsb.sort_values(["Tail", "timestamp"]).reset_index(drop=True)

# Previous ping values per aircraft (shift within group)
df_adsb["prev_range_nm"]  = df_adsb.groupby("Tail")["range_nm"].shift(1)
df_adsb["prev_altitude"]  = df_adsb.groupby("Tail")["Altitude"].shift(1)
df_adsb["prev_timestamp"] = df_adsb.groupby("Tail")["timestamp"].shift(1)

# Instantaneous deltas
df_adsb["range_delta_nm"] = df_adsb["range_nm"] - df_adsb["prev_range_nm"]
df_adsb["alt_delta_ft"]   = df_adsb["Altitude"] - df_adsb["prev_altitude"]

# Time delta in seconds (for rate calculations)
df_adsb["time_delta_s"] = (
    df_adsb["timestamp"] - df_adsb["prev_timestamp"]
).dt.total_seconds()

# Smooth approach/descent over a rolling window per aircraft
# Single ping deltas are noisy — use 3-ping rolling mean
# 📚 Rolling mean smooths out noisy individual measurements.
# window=3 means average the current and 2 previous values.
# min_periods=1 prevents NaN when fewer than 3 pings exist.
df_adsb["range_delta_smooth"] = (
    df_adsb.groupby("Tail")["range_delta_nm"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)
df_adsb["alt_delta_smooth"] = (
    df_adsb.groupby("Tail")["alt_delta_ft"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# Boolean flags
df_adsb["is_approaching"] = df_adsb["range_delta_smooth"] < -0.02  # closing > 0.02 NM/ping
df_adsb["is_departing"]   = df_adsb["range_delta_smooth"] >  0.02  # opening > 0.02 NM/ping
df_adsb["is_descending"]  = df_adsb["alt_delta_smooth"]   < -20    # losing > 20 ft/ping
df_adsb["is_climbing"]    = df_adsb["alt_delta_smooth"]   >  20    # gaining > 20 ft/ping

# AGL (altitude above ground level)
df_adsb["agl"] = df_adsb["Altitude"] - KBTP["elevation"]

# Runway alignment: angle between aircraft heading and active runway heading
# "Aligned" means heading within ±50° of runway heading (for inbound)
df_adsb["hdg_diff_rwy"] = df_adsb["Heading"].apply(
    lambda hdg: abs(((hdg - active_runway["hdg"]) + 180) % 360 - 180)
    if pd.notna(hdg) else 180
)
df_adsb["is_runway_aligned"] = df_adsb["hdg_diff_rwy"] <= 50

# 45-degree pattern entry check
# Aircraft approaching from 45° to the downwind leg
# Downwind heading = runway heading + 180°
downwind_hdg = (active_runway["hdg"] + 180) % 360
df_adsb["hdg_diff_45entry"] = df_adsb["Heading"].apply(
    lambda hdg: abs(((hdg - (downwind_hdg - 45)) + 180) % 360 - 180)
    if pd.notna(hdg) else 180
)
df_adsb["is_45deg_entry"] = df_adsb["hdg_diff_45entry"] <= 30

print("✅ Track features computed")
print(f"\nApproaching aircraft pings : {df_adsb['is_approaching'].sum():,}")
print(f"Departing  aircraft pings : {df_adsb['is_departing'].sum():,}")
print(f"Descending aircraft pings : {df_adsb['is_descending'].sum():,}")
print(f"Runway-aligned pings      : {df_adsb['is_runway_aligned'].sum():,}")

# Quick sanity check — show a specific aircraft's track features
print(f"\nSample track features for N8275D (the problematic aircraft):")
n8275d = df_adsb[df_adsb["Tail"] == "N8275D"][
    ["timestamp","Altitude","agl","range_nm","range_delta_nm",
     "alt_delta_ft","is_approaching","is_descending","hdg_diff_rwy"]
].head(15)
if len(n8275d) > 0:
    display(n8275d)
else:
    print("  N8275D not found — check tail number spelling")

Computing track-level features...
✅ Track features computed

Approaching aircraft pings : 36,007
Departing  aircraft pings : 39,164
Descending aircraft pings : 13,782
Runway-aligned pings      : 63,258

Sample track features for N8275D (the problematic aircraft):


,timestamp,Altitude,agl,range_nm,range_delta_nm,alt_delta_ft,is_approaching,is_descending,hdg_diff_rwy
171395,2020-10-22 14:28:54.667999,3775.0,2527.0,12.439222,NaN,NaN,False,False,12.0
171396,2020-10-22 14:28:55.904000,3775.0,2527.0,12.439222,0.000000,0.0,False,False,24.0
171397,2020-10-22 14:28:57.036000,3800.0,2552.0,12.370729,-0.068492,25.0,True,False,24.0
171398,2020-10-22 14:28:57.036000,3800.0,2552.0,12.370729,0.000000,0.0,True,False,24.0
171399,2020-10-22 14:28:58.899000,3800.0,2552.0,12.370729,0.000000,0.0,True,False,24.0
171400,2020-10-22 14:29:00.794000,3825.0,2577.0,12.265703,-0.105027,25.0,True,False,24.0
171401,2020-10-22 14:29:01.828000,3825.0,2577.0,12.265703,0.000000,0.0,True,False,27.0
171402,2020-10-22 14:29:02.862000,3850.0,2602.0,12.208301,-0.057402,25.0,True,False,27.0
171403,2020-10-22 14:29:03.459999,3850.0,2602.0,12.191220,-0.017081,0.0,True,False,28.0
171404,2020-10-22 14:29:04.586000,3875.0,2627.0,12.159703,-0.031517,25.0,True,False,28.0


In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC: Why did DEPARTING jump from 3,735 to 18,787?
# ═══════════════════════════════════════════════════════════════════════════════

print("DEPARTING SPIKE DIAGNOSTIC")
print("="*60)

df_departing = df_adsb[df_adsb["phase"] == "DEPARTING"].copy()

print(f"\nTotal DEPARTING pings: {len(df_departing):,}")
print(f"\nAltitude AGL distribution:")
agl_bins = [0, 200, 500, 1000, 2000, 3000, 5000, 99999]
agl_labels = ["0–200", "200–500", "500–1k", "1k–2k", "2k–3k", "3k–5k", "5k+"]
df_departing["agl_bin"] = pd.cut(df_departing["agl"], bins=agl_bins, labels=agl_labels)
print(df_departing["agl_bin"].value_counts().sort_index().to_string())

print(f"\nRange distribution:")
rng_bins = [0, 1, 3, 5, 8, 12, 20, 999]
rng_labels = ["0–1nm","1–3nm","3–5nm","5–8nm","8–12nm","12–20nm","20+nm"]
df_departing["rng_bin"] = pd.cut(df_departing["range_nm"], bins=rng_bins, labels=rng_labels)
print(df_departing["rng_bin"].value_counts().sort_index().to_string())

print(f"\nTop 15 aircraft by DEPARTING ping count:")
display(
    df_departing.groupby("Tail").size()
    .sort_values(ascending=False).head(15)
    .reset_index(name="count")
)

# Look at specific suspicious cases
# Aircraft classified DEPARTING but far from airport or at wrong altitude
suspicious = df_departing[
    (df_departing["range_nm"] > 8) |    # too far to be "departing"
    (df_departing["agl"] > 3000)        # too high for a departure
].copy()

print(f"\nSuspicious DEPARTING pings (rng>8nm OR agl>3000ft): {len(suspicious):,}")
if len(suspicious) > 0:
    print(f"These should probably be TRANSITING")
    print(f"\nSample suspicious cases:")
    display(suspicious[["Tail","timestamp","Altitude","agl",
                         "range_nm","Heading","hdg_diff_rwy",
                         "is_departing","is_climbing"]].head(10))

# The real question: is_departing=True but NOT near airport
# Our bug: step 6 catches aircraft that are "moving away" (is_departing=True)
# at low altitude anywhere — but low + moving away could just be pattern
# aircraft between legs where range briefly increases
print(f"\nDEPARTING pings where is_departing=False (caught by hdg_rwy logic):")
false_dept = df_departing[~df_departing["is_departing"]]
print(f"  Count: {len(false_dept):,}")
print(f"  These are caught by step 4 (runway-aligned + climbing)")
print(f"\nDEPARTING pings where is_departing=True (caught by step 6):")
true_dept = df_departing[df_departing["is_departing"]]
print(f"  Count: {len(true_dept):,}")
print(f"  Range stats: min={true_dept['range_nm'].min():.1f}  "
      f"max={true_dept['range_nm'].max():.1f}  "
      f"median={true_dept['range_nm'].median():.1f} nm")

DEPARTING SPIKE DIAGNOSTIC

Total DEPARTING pings: 18,787

Altitude AGL distribution:
agl_bin
0–200         10
200–500      415
500–1k      1989
1k–2k      10084
2k–3k       4924
3k–5k       1362
5k+            0

Range distribution:
rng_bin
0–1nm        98
1–3nm       358
3–5nm      4072
5–8nm      5137
8–12nm     6130
12–20nm    2992
20+nm         0

Top 15 aircraft by DEPARTING ping count:


,Tail,count
0,N53226,1036
1,N600HJ,925
2,N8275D,760
3,N819CM,742
4,N56987,673
5,N1452U,624
6,N24TS,539
7,N191ND,511
8,N92141,477
9,N41548,472



Suspicious DEPARTING pings (rng>8nm OR agl>3000ft): 9,569
These should probably be TRANSITING

Sample suspicious cases:


,Tail,timestamp,Altitude,agl,range_nm,Heading,hdg_diff_rwy,is_departing,is_climbing
3835,CAP3733,2020-10-22 21:16:01.137999,2225.0,977.0,9.013823,309.0,123.0,True,False
3836,CAP3733,2020-10-22 21:16:02.201999,2250.0,1002.0,8.962883,307.0,125.0,True,False
3837,CAP3733,2020-10-22 21:16:03.181999,2250.0,1002.0,8.962883,307.0,125.0,True,False
5563,CPA8800,2020-10-22 14:03:10.292999,4275.0,3027.0,12.707009,123.0,51.0,True,False
5564,CPA8800,2020-10-22 14:03:11.432999,4275.0,3027.0,12.735012,130.0,58.0,True,False
5565,CPA8800,2020-10-22 14:03:12.462999,4275.0,3027.0,12.735012,130.0,58.0,True,False
5567,CPA8800,2020-10-22 14:03:14.653000,4250.0,3002.0,12.841367,132.0,60.0,True,False
5568,CPA8800,2020-10-22 14:03:16.013000,4225.0,2977.0,12.912038,136.0,64.0,True,False
5569,CPA8800,2020-10-22 14:03:17.033000,4225.0,2977.0,12.955793,136.0,64.0,True,False
5570,CPA8800,2020-10-22 14:03:18.422999,4200.0,2952.0,12.997363,146.0,74.0,True,False



DEPARTING pings where is_departing=False (caught by hdg_rwy logic):
  Count: 531
  These are caught by step 4 (runway-aligned + climbing)

DEPARTING pings where is_departing=True (caught by step 6):
  Count: 18,256
  Range stats: min=0.7  max=15.0  median=8.0 nm


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FLIGHT PHASE CLASSIFIER
#
# 📚 Concept — Enum:
#   An Enum (enumeration) is a set of named constants. Using an Enum instead of
#   raw strings ("FINAL", "DOWNWIND") prevents typos and makes code
#   self-documenting. MyPhase.FINAL is always correct; "FIANL" (typo) passes
#   silently. This is defensive programming — a key professional habit.
# ═══════════════════════════════════════════════════════════════════════════════

class FlightPhase(Enum):
    UNKNOWN        = auto()
    TRANSITING     = auto()   # overflying, not airport-bound
    INBOUND_10NM   = auto()   # 10–19 NM from airport, descending/level
    INBOUND_5NM    = auto()   # 5–10 NM from airport
    INBOUND_3NM    = auto()   # 3–5 NM, likely straight-in
    PATTERN_ENTRY  = auto()   # entering the traffic pattern
    CROSSWIND      = auto()
    DOWNWIND       = auto()
    BASE           = auto()
    FINAL          = auto()
    LANDING        = auto()   # on ground or very low
    TAXIING        = auto()
    DEPARTING      = auto()   # climbing away
    DEPARTED       = auto()   # left the area

def classify_phase(row: pd.Series, airport: dict,
                   active_runway: dict) -> FlightPhase:
    """
    Classify a single ADS-B ping into a FlightPhase.
    
    Logic hierarchy (order matters — check most specific first):
    1. On ground / very low → TAXIING or LANDING
    2. In traffic pattern → pattern leg
    3. Climbing away from airport → DEPARTING
    4. Approaching (inbound) by range band → INBOUND_X
    5. High altitude overflying → TRANSITING
    """
    # Guard against NaN values
    if pd.isna(row.get("Lat")) or pd.isna(row.get("Altitude")):
        return FlightPhase.UNKNOWN
    
    rng_km   = float(row.get("Range",   999))
    rng_nm   = km_to_nm(rng_km)
    alt_msl  = float(row.get("Altitude", 0))
    speed    = float(row.get("Speed",    0))
    hdg      = float(row.get("Heading",  0))
    lat      = float(row["Lat"])
    lon      = float(row["Lon"])
    agl      = alt_msl - airport["elevation"]
    
    # ── 1. Ground / very low ──────────────────────────────────────────────────
    if agl < 100 and speed < 50:
        return FlightPhase.TAXIING
    if agl < 200 and rng_nm < 1.0:
        return FlightPhase.LANDING
    
    # ── 2. Traffic pattern ────────────────────────────────────────────────────
    pattern_leg = classify_pattern_leg(
        lat, lon, hdg, alt_msl,
        airport["lat"], airport["lon"],
        active_runway, airport["elevation"]
    )
    if pattern_leg == "FINAL":     return FlightPhase.FINAL
    if pattern_leg == "BASE":      return FlightPhase.BASE
    if pattern_leg == "DOWNWIND":  return FlightPhase.DOWNWIND
    if pattern_leg == "CROSSWIND": return FlightPhase.CROSSWIND
    
    # ── 3. Departing — climbing away from airport ─────────────────────────────
    rwy_hdg = active_runway["hdg"]
    hdg_diff_upwind = abs(((hdg - rwy_hdg) + 180) % 360 - 180)
    if hdg_diff_upwind <= 40 and agl > 100 and agl < 3000 and rng_nm < 5:
        return FlightPhase.DEPARTING
    
    # ── 4. Inbound by range band ──────────────────────────────────────────────
    # High-altitude jets transiting → altitude cutoff
    if alt_msl > 8000 and rng_nm > 5:
        return FlightPhase.TRANSITING
    
    if rng_nm <= 3:
        return FlightPhase.INBOUND_3NM
    elif rng_nm <= 5.5:   # ~5 NM with tolerance
        return FlightPhase.INBOUND_5NM
    elif rng_nm <= 11:    # ~10 NM with tolerance
        return FlightPhase.INBOUND_10NM
    elif rng_nm > 11:
        return FlightPhase.TRANSITING
    
    return FlightPhase.UNKNOWN


# ═══════════════════════════════════════════════════════════════════════════════
# IMPROVED FLIGHT PHASE CLASSIFIER v2
#
# Key improvements over v1:
#   1. Inbound/outbound discrimination using range_delta_smooth
#   2. Altitude profile check — inbound must be descending or at pattern alt
#   3. AGL constraints for each phase
#   4. Runway alignment check for INBOUND phases
#   5. Pattern altitude window: 800–1500 ft AGL (props + high-performance)
#   6. Proper TRANSITING definition: high altitude OR not airport-bound
# ═══════════════════════════════════════════════════════════════════════════════

# Pattern altitude constants (FAA standard)
PATTERN_ALT_MIN_AGL = 700    # ft — bottom of pattern window (allow some tolerance)
PATTERN_ALT_MAX_AGL = 1600   # ft — top (1500 AGL for high-performance + margin)
APPROACH_ALT_MAX_AGL = 4000  # ft — above this at close range = not landing
TRANSIT_ALT_MIN_MSL  = 6000  # ft MSL — above this = likely IFR/transit


def classify_phase_v2(row: pd.Series, airport: dict,
                       active_runway: dict) -> FlightPhase:
    """
    Improved flight phase classifier using track-level features.
    
    Decision tree (order = priority):
    
    1. Data quality guard
    2. On ground → TAXIING
    3. Very low and slow near airport → LANDING roll
    4. In traffic pattern (strict: altitude + heading + range)
    5. Climbing away on runway heading → DEPARTING
    6. INBOUND — must satisfy ALL of:
       a. is_approaching (range decreasing)
       b. Reasonable altitude for approach (not 5000ft at 3nm)
       c. Within inbound range band
    7. OUTBOUND — departing the area (range increasing, climbing or level)
    8. High altitude → TRANSITING
    9. Everything else → TRANSITING (overflight, unknown)
    """
    # ── Guard ─────────────────────────────────────────────────────────────────
    try:
        rng_nm   = float(row["range_nm"])
        alt_msl  = float(row["Altitude"])
        speed    = float(row.get("Speed", 0))
        hdg      = float(row.get("Heading", 0))
        agl      = float(row["agl"])
        is_appr  = bool(row["is_approaching"])
        is_dept  = bool(row["is_departing"])
        is_desc  = bool(row["is_descending"])
        is_climb = bool(row["is_climbing"])
        hdg_rwy  = float(row["hdg_diff_rwy"])   # degrees off runway heading
        rwy_hdg  = active_runway["hdg"]
    except (ValueError, KeyError, TypeError):
        return FlightPhase.UNKNOWN

    # ── 1. On ground ──────────────────────────────────────────────────────────
    if agl < 50 and speed < 35:
        return FlightPhase.TAXIING

    # ── 2. Very close to airport, very low, slow → landing rollout ───────────
    if agl < 150 and rng_nm < 0.8 and speed < 80:
        return FlightPhase.LANDING

    # ── 3. Traffic pattern — strict altitude + heading + proximity ────────────
    # Must be: within 5 NM, near pattern altitude (700–1600 AGL)
    if rng_nm <= 5.0 and PATTERN_ALT_MIN_AGL <= agl <= PATTERN_ALT_MAX_AGL:

        # Final approach: aligned with runway, descending or low, within 3 NM
        if (hdg_rwy <= 30 and rng_nm <= 3.5 and
                (is_desc or agl < 1000)):
            return FlightPhase.FINAL

        # Base leg: ~90° to runway heading, within 3 NM
        hdg_diff_base = abs(((hdg - (rwy_hdg + 270)) + 180) % 360 - 180)
        if hdg_diff_base <= 35 and rng_nm <= 3.0:
            return FlightPhase.BASE

        # Downwind: ~180° opposite runway heading, 0.5–2.5 NM from centreline
        hdg_diff_dw = abs(((hdg - (rwy_hdg + 180)) + 180) % 360 - 180)
        if hdg_diff_dw <= 35 and 0.5 <= rng_nm <= 3.0:
            return FlightPhase.DOWNWIND

        # Crosswind: ~90° from runway heading
        hdg_diff_cw = abs(((hdg - (rwy_hdg + 90)) + 180) % 360 - 180)
        if hdg_diff_cw <= 35 and rng_nm <= 2.5:
            return FlightPhase.CROSSWIND

        # 45-degree pattern entry (approaching downwind from outside)
        if bool(row.get("is_45deg_entry", False)) and 1.5 <= rng_nm <= 4.0:
            return FlightPhase.PATTERN_ENTRY

    # ── 4. Departing — climbing away on runway heading ────────────────────────
    # Aligned with runway heading, climbing, close to airport
    if (hdg_rwy <= 40 and is_climb and
            rng_nm <= 6.0 and agl < 3000):
        return FlightPhase.DEPARTING

    # ── 5. INBOUND — approaching, at reasonable approach altitude ─────────────
    # Core logic: aircraft must be getting CLOSER (is_approaching)
    # AND altitude must be plausible for an approach at that range
    #
    # 📚 Altitude-range profile for a 3° glidepath:
    #   At 10 NM → ~3,000 ft AGL (10 × 300 ft/NM at 3° descent)
    #   At  5 NM → ~1,500 ft AGL
    #   At  3 NM → ~  900 ft AGL
    #   We allow 2× this as upper bound to catch early descents
    #   Formula: max_agl = range_nm × 600 ft/NM (generous 6° envelope)

    max_reasonable_agl = rng_nm * 600   # generous upper bound

    if is_appr and agl <= max_reasonable_agl:

        if 8.5 <= rng_nm <= 13:
            return FlightPhase.INBOUND_10NM

        elif 4.5 <= rng_nm <= 8.5:
            return FlightPhase.INBOUND_5NM

        elif 1.5 <= rng_nm < 4.5:
            return FlightPhase.INBOUND_3NM

    # ── 6. OUTBOUND — moving away from airport ───────────────────────────────
    if is_dept and rng_nm < 15:
        # True departure: climbing AND runway-aligned AND close to airport
        if is_climb and hdg_rwy <= 50 and rng_nm <= 8:
            return FlightPhase.DEPARTING
        # Already high and moving away → transiting
        return FlightPhase.TRANSITING

    # ── 7. High altitude → definitely transiting ─────────────────────────────
    if alt_msl > TRANSIT_ALT_MIN_MSL:
        return FlightPhase.TRANSITING

    # ── 8. Default — not enough context ──────────────────────────────────────
    return FlightPhase.TRANSITING


# ── Quick validation on the N8275D case ──────────────────────────────────────
print("CLASSIFIER v2 VALIDATION")
print("="*55)

# Test the specific problematic case you identified
problem_cases = df_adsb[
    (df_adsb["Tail"] == "N8275D") &
    (df_adsb["range_nm"] < 3) &
    (df_adsb["Altitude"] > 3000)
].head(5)

if len(problem_cases) > 0:
    print(f"\nN8275D high-altitude close-range pings:")
    for _, row in problem_cases.iterrows():
        phase_v1 = classify_phase(row, KBTP, active_runway).name
        phase_v2 = classify_phase_v2(row, KBTP, active_runway).name
        max_agl  = row["range_nm"] * 600
        print(f"  alt={row['Altitude']:.0f}ft  agl={row['agl']:.0f}ft  "
              f"rng={row['range_nm']:.1f}nm  "
              f"max_reasonable_agl={max_agl:.0f}ft")
        print(f"  v1={phase_v1}  →  v2={phase_v2}")
        print(f"  is_approaching={row['is_approaching']}  "
              f"hdg_diff_rwy={row['hdg_diff_rwy']:.0f}°")
else:
    print("N8275D: no problematic pings found (or tail not in dataset)")

# Test a clean set
print(f"\nSample classifications (close-range aircraft only):")
sample = df_adsb[df_adsb["range_nm"] < 12].sample(12, random_state=99)
for _, row in sample.iterrows():
    p1 = classify_phase(row, KBTP, active_runway).name
    p2 = classify_phase_v2(row, KBTP, active_runway).name
    changed = "← CHANGED" if p1 != p2 else ""
    print(f"  {str(row['Tail']):10s}  rng={row['range_nm']:5.1f}nm  "
          f"agl={row['agl']:5.0f}ft  "
          f"appr={str(row['is_approaching']):5s}  "
          f"v1={p1:15s} → v2={p2:15s} {changed}")

CLASSIFIER v2 VALIDATION

N8275D high-altitude close-range pings:
  alt=5225ft  agl=3977ft  rng=3.0nm  max_reasonable_agl=1777ft
  v1=INBOUND_3NM  →  v2=TRANSITING
  is_approaching=True  hdg_diff_rwy=16°
  alt=5225ft  agl=3977ft  rng=2.9nm  max_reasonable_agl=1758ft
  v1=INBOUND_3NM  →  v2=TRANSITING
  is_approaching=True  hdg_diff_rwy=16°
  alt=5225ft  agl=3977ft  rng=2.9nm  max_reasonable_agl=1740ft
  v1=INBOUND_3NM  →  v2=TRANSITING
  is_approaching=True  hdg_diff_rwy=16°
  alt=5225ft  agl=3977ft  rng=2.9nm  max_reasonable_agl=1718ft
  v1=INBOUND_3NM  →  v2=TRANSITING
  is_approaching=True  hdg_diff_rwy=16°
  alt=5225ft  agl=3977ft  rng=2.8nm  max_reasonable_agl=1699ft
  v1=INBOUND_3NM  →  v2=TRANSITING
  is_approaching=True  hdg_diff_rwy=16°

Sample classifications (close-range aircraft only):
  N81PA       rng= 10.6nm  agl=11027ft  appr=False  v1=TRANSITING      → v2=TRANSITING      
  N41548      rng=  9.4nm  agl= 1452ft  appr=True   v1=INBOUND_10NM    → v2=INBOUND_10NM    
  N66

In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALLOUT TEMPLATE ENGINE
#
# This is the core NLP component: structured text generation from structured data
# Pattern: "WHO_CALLING, WHO_WE_ARE, WHERE_WE_ARE, INTENTION, WHO_CALLING"
#
# 📚 Concept — Template-based NLG (Natural Language Generation):
#   The simplest form of NLG is template filling: you define sentence patterns
#   with slots, and fill them from data. This is how weather forecasts, stock
#   alerts, and sports summaries were generated before LLMs. It's deterministic,
#   auditable, and correct — exactly what aviation safety requires.
#   Compare with neural NLG (GPT-style): creative but unpredictable.
#   For safety-critical ground truth labels, rule-based is the right choice.
# ═══════════════════════════════════════════════════════════════════════════════

# Phonetic alphabet for tail number spelling
# "N819CM" → "November 8 1 9 Charlie Mike"
PHONETIC = {
    "A": "Alpha",   "B": "Bravo",   "C": "Charlie", "D": "Delta",
    "E": "Echo",    "F": "Foxtrot", "G": "Golf",    "H": "Hotel",
    "I": "India",   "J": "Juliet",  "K": "Kilo",    "L": "Lima",
    "M": "Mike",    "N": "November","O": "Oscar",   "P": "Papa",
    "Q": "Quebec",  "R": "Romeo",   "S": "Sierra",  "T": "Tango",
    "U": "Uniform", "V": "Victor",  "W": "Whiskey", "X": "X-ray",
    "Y": "Yankee",  "Z": "Zulu",
}

def tail_to_phonetic(tail: str) -> str:
    """
    Convert tail number to phonetic radio speech.
    FAA rules: letters → phonetic alphabet, digits → spoken as digits.
    First character 'N' (US registration) → "November"
    Last 3 characters used for abbreviated callouts, full for initial.
    
    Example: "N819CM" → "November 8 1 9 Charlie Mike"
    """
    if not tail or pd.isna(tail):
        return "Unknown Traffic"
    tail = str(tail).upper().strip()
    parts = []
    for char in tail:
        if char.isalpha():
            parts.append(PHONETIC.get(char, char))
        elif char.isdigit():
            parts.append(char)   # digits spoken individually
    return " ".join(parts)


def round_to_nearest_5nm(range_nm: float) -> int:
    """
    Round range to nearest standard callout distance.
    Pilots say "10 miles", "5 miles", "3 miles" — not "7.3 miles".
    
    📚 This is called quantisation — mapping continuous values to
       discrete categories. Standard aviation callout distances are
       10, 5, and 3 NM for inbound traffic.
    """
    if range_nm >= 8:   return 10
    elif range_nm >= 4: return 5
    else:               return 3


def generate_callout(tail: str, phase: FlightPhase,
                     range_nm: float, bearing_deg: float,
                     active_runway: dict, airport: dict) -> Optional[str]:
    """
    Generate the expected CTAF radio callout for a given aircraft state.
    
    Returns the full callout string, or None if no callout expected
    (e.g., transiting overflight, taxiing, unknown phase).
    
    The 4-W structure:
        [WHO_CALLING] = "[Airport] Traffic"
        [WHO_WE_ARE]  = "[Tail phonetic]"
        [WHERE_WE_ARE] = "[distance] [direction]" or "[pattern leg]"
        [INTENTION]   = "inbound runway [N]" or "turning [leg]" or "on final"
        [WHO_CALLING] = "[Airport] Traffic"
    """
    airport_name = airport["name"]   # "Butler"
    rwy_id       = active_runway["id"]  # "26"
    who_calling  = f"{airport_name} Traffic"
    who_we_are   = tail_to_phonetic(tail)
    
    # ── Generate position and intention strings per phase ─────────────────────
    
    if phase in (FlightPhase.INBOUND_10NM,
                 FlightPhase.INBOUND_5NM,
                 FlightPhase.INBOUND_3NM):
        dist_nm     = round_to_nearest_5nm(range_nm)
        direction   = bearing_to_compass(bearing_deg)
        
        if phase == FlightPhase.INBOUND_3NM:
            where  = f"{dist_nm} miles {direction}, straight-in"
            intent = f"runway {rwy_id}"
        else:
            where  = f"{dist_nm} miles {direction}"
            intent = f"inbound runway {rwy_id}"
        
        return (f"{who_calling}, {who_we_are}, "
                f"{where}, {intent}, "
                f"{who_calling}")
    
    elif phase == FlightPhase.DOWNWIND:
        where  = f"left downwind"
        intent = f"runway {rwy_id}"
        return (f"{who_calling}, {who_we_are}, "
                f"{where}, {intent}, "
                f"{who_calling}")
    
    elif phase == FlightPhase.BASE:
        return (f"{who_calling}, {who_we_are}, "
                f"turning base, runway {rwy_id}, "
                f"{who_calling}")
    
    elif phase == FlightPhase.FINAL:
        return (f"{who_calling}, {who_we_are}, "
                f"final, runway {rwy_id}, "
                f"{who_calling}")
    
    elif phase == FlightPhase.DEPARTING:
        return (f"{who_calling}, {who_we_are}, "
                f"departing runway {rwy_id}, "
                f"leaving the area, "
                f"{who_calling}")
    
    # No callout needed for transiting, taxiing, landing roll
    return None


# ── Quick test ────────────────────────────────────────────────────────────────
test_cases = [
    ("N819CM",  FlightPhase.INBOUND_10NM, 9.5,  15.0),   # 10mi North inbound
    ("N532261", FlightPhase.INBOUND_5NM,  4.8, 270.0),   # 5mi West
    ("N921419", FlightPhase.INBOUND_3NM,  2.8, 260.0),   # 3mi straight-in
    ("EJM410",  FlightPhase.DOWNWIND,     1.2, 170.0),   # on downwind
    ("N410LG",  FlightPhase.FINAL,        0.8, 258.0),   # on final 26
    ("N6886D",  FlightPhase.DEPARTING,    1.5,  78.0),   # departing
]

print("CALLOUT GENERATION TEST\n" + "="*70)
for tail, phase, rng, brg in test_cases:
    callout = generate_callout(tail, phase, rng, brg, active_runway, KBTP)
    print(f"\n  Aircraft : {tail}")
    print(f"  Phase    : {phase.name}")
    print(f"  Range    : {rng:.1f} NM  Bearing: {brg:.0f}°")
    print(f"  CALLOUT  : {callout}")

CALLOUT GENERATION TEST

  Aircraft : N819CM
  Phase    : INBOUND_10NM
  Range    : 9.5 NM  Bearing: 15°
  CALLOUT  : Butler Traffic, November 8 1 9 Charlie Mike, 10 miles North, inbound runway 08, Butler Traffic

  Aircraft : N532261
  Phase    : INBOUND_5NM
  Range    : 4.8 NM  Bearing: 270°
  CALLOUT  : Butler Traffic, November 5 3 2 2 6 1, 5 miles West, inbound runway 08, Butler Traffic

  Aircraft : N921419
  Phase    : INBOUND_3NM
  Range    : 2.8 NM  Bearing: 260°
  CALLOUT  : Butler Traffic, November 9 2 1 4 1 9, 3 miles West, straight-in, runway 08, Butler Traffic

  Aircraft : EJM410
  Phase    : DOWNWIND
  Range    : 1.2 NM  Bearing: 170°
  CALLOUT  : Butler Traffic, Echo Juliet Mike 4 1 0, left downwind, runway 08, Butler Traffic

  Aircraft : N410LG
  Phase    : FINAL
  Range    : 0.8 NM  Bearing: 258°
  CALLOUT  : Butler Traffic, November 4 1 0 Lima Golf, final, runway 08, Butler Traffic

  Aircraft : N6886D
  Phase    : DEPARTING
  Range    : 1.5 NM  Bearing: 78°
  CALLO

In [29]:
print("Applying classifier v2 to full dataset...")

# Apply improved classifier
df_adsb["phase"] = df_adsb.apply(
    lambda row: classify_phase_v2(row, KBTP, active_runway).name,
    axis=1
)

# Re-generate callouts
df_adsb["expected_callout"] = df_adsb.apply(
    lambda row: generate_callout(
        tail          = row["Tail"],
        phase         = FlightPhase[row["phase"]],
        range_nm      = row["range_nm"],
        bearing_deg   = row["bearing_from_airport"],
        active_runway = active_runway,
        airport       = KBTP,
    ) if pd.notna(row["range_nm"]) else None,
    axis=1
)

# ── Compare v1 vs v2 distributions ───────────────────────────────────────────
print("\nPhase distribution comparison (v1 → v2):")
print(f"{'Phase':<16} {'v1 count':>10}  {'v2 count':>10}  {'change':>10}")
print("-" * 52)

phase_v2 = df_adsb["phase"].value_counts()

# v1 reference numbers from your previous run
v1_ref = {
    "TRANSITING":   77367, "INBOUND_10NM": 52372,
    "FINAL":        15474, "TAXIING":      15131,
    "INBOUND_5NM":  14641, "LANDING":      13742,
    "INBOUND_3NM":  11779, "BASE":          7012,
    "CROSSWIND":     4451, "DOWNWIND":      3918,
    "DEPARTING":     3735,
}

all_phases = sorted(set(list(v1_ref.keys()) + list(phase_v2.index)))
for phase in all_phases:
    v1 = v1_ref.get(phase, 0)
    v2 = phase_v2.get(phase, 0)
    delta = v2 - v1
    bar = "█" * int(v2 / 2000)
    sign = "+" if delta >= 0 else ""
    print(f"  {phase:<16}: {v1:>8,}  → {v2:>8,}  ({sign}{delta:,})")

n_callouts = df_adsb["expected_callout"].notna().sum()
print(f"\nCallouts generated: {n_callouts:,} ({100*n_callouts/len(df_adsb):.1f}%)")
print(f"\nExpected: INBOUND_10NM should DROP significantly")
print(f"  (many of those were high-altitude transits falsely classified as inbound)")

# Save updated version
out_path = OUTPUT_DIR / "adsb_with_callouts_2020-10-22.csv"
df_adsb.to_csv(out_path, index=False)
print(f"\n✅ Saved → {out_path}")

# ── v2.1 expected targets ────────────────────────────────────────────────────
print("\nExpected after v2.1 fix:")
print("  DEPARTING    : should drop from 18,787 → closer to 4,000–8,000")
print("  TRANSITING   : should increase slightly (absorbing false departures)")  
print("  All others   : should be stable")
print("\nKey sanity checks:")
print(f"  FINAL / LANDING ratio should be ~1:1 or FINAL > LANDING")
print(f"  (every landing is preceded by a final approach)")

finals   = phase_v2.get("FINAL",   0)
landings = phase_v2.get("LANDING", 0)
ratio    = finals / landings if landings > 0 else 0
print(f"  Current: FINAL={finals:,}  LANDING={landings:,}  ratio={ratio:.2f}")
if ratio < 0.5:
    print(f"  ⚠  Too few FINALs relative to LANDINGs — final approach too strict")
elif ratio > 3:
    print(f"  ⚠  Too many FINALs relative to LANDINGs — investigate")
else:
    print(f"  ✅ Ratio looks reasonable")

Applying classifier v2 to full dataset...

Phase distribution comparison (v1 → v2):
Phase              v1 count    v2 count      change
----------------------------------------------------
  BASE            :    7,012  →    2,946  (-4,066)
  CROSSWIND       :    4,451  →    2,642  (-1,809)
  DEPARTING       :    3,735  →    1,004  (-2,731)
  DOWNWIND        :    3,918  →    1,458  (-2,460)
  FINAL           :   15,474  →    6,919  (-8,555)
  INBOUND_10NM    :   52,372  →    5,288  (-47,084)
  INBOUND_3NM     :   11,779  →    3,839  (-7,940)
  INBOUND_5NM     :   14,641  →    5,979  (-8,662)
  LANDING         :   13,742  →   11,692  (-2,050)
  PATTERN_ENTRY   :        0  →      626  (+626)
  TAXIING         :   15,131  →   13,318  (-1,813)
  TRANSITING      :   77,367  →  163,911  (+86,544)

Callouts generated: 27,433 (12.5%)

Expected: INBOUND_10NM should DROP significantly
  (many of those were high-altitude transits falsely classified as inbound)

✅ Saved → C:\xcas-ga-comms-assistant

In [30]:
# Show aircraft positions coloured by phase, with callout annotations
# Focus on a 1-hour window when most aircraft are active

window_start = pd.Timestamp("2020-10-22 14:00:00")
window_end   = pd.Timestamp("2020-10-22 15:00:00")

df_window = df_adsb[
    (df_adsb["timestamp"] >= window_start) &
    (df_adsb["timestamp"] <= window_end)  &
    (df_adsb["range_nm"]  <= 15)
].copy()

phase_colours = {
    "INBOUND_10NM"  : "#FF6B35",
    "INBOUND_5NM"   : "#FFD166",
    "INBOUND_3NM"   : "#FF4444",
    "DOWNWIND"      : "#00B4D8",
    "BASE"          : "#48CAE4",
    "FINAL"         : "#00FF88",
    "DEPARTING"     : "#A8DADC",
    "TAXIING"       : "#AAAAAA",
    "TRANSITING"    : "#334466",
    "UNKNOWN"       : "#222222",
    "LANDING"       : "#FFFFFF",
}

fig = go.Figure()

for phase_name, grp in df_window.groupby("phase"):
    colour = phase_colours.get(phase_name, "#888888")
    has_callout = grp["expected_callout"].notna()
    
    # Pings WITH callouts → markers + hover
    if has_callout.any():
        g = grp[has_callout]
        fig.add_trace(go.Scattermap(
            lat=g["Lat"].tolist(),
            lon=g["Lon"].tolist(),
            mode="markers",
            name=phase_name,
            marker=dict(size=7, color=colour),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Time: %{customdata[1]}<br>"
                "Phase: %{customdata[2]}<br>"
                "Range: %{customdata[3]:.1f} NM<br>"
                "<i>%{customdata[4]}</i><extra></extra>"
            ),
            customdata=list(zip(
                g["Tail"],
                g["timestamp"].dt.strftime("%H:%M:%S"),
                g["phase"],
                g["range_nm"],
                g["expected_callout"].fillna("—"),
            )),
        ))

# Airport marker
fig.add_trace(go.Scattermap(
    lat=[KBTP["lat"]], lon=[KBTP["lon"]],
    mode="markers+text",
    marker=dict(size=18, color="yellow"),
    text=["KBTP"], textposition="top right",
    textfont=dict(color="yellow", size=12),
    name="KBTP",
    hoverinfo="skip",
))

# Range rings
def make_ring(lat_c, lon_c, r_nm, n=180):
    R, r_km = 6371, r_nm * 1.852
    lats, lons = [], []
    for i in range(n+1):
        b = math.radians(i * 360 / n)
        d = r_km / R
        la1, lo1 = math.radians(lat_c), math.radians(lon_c)
        la2 = math.asin(math.sin(la1)*math.cos(d)+math.cos(la1)*math.sin(d)*math.cos(b))
        lo2 = lo1 + math.atan2(math.sin(b)*math.sin(d)*math.cos(la1),
                                math.cos(d)-math.sin(la1)*math.sin(la2))
        lats.append(math.degrees(la2)); lons.append(math.degrees(lo2))
    return lats, lons

for r_nm, colour in [(10,"rgba(255,100,100,0.5)"),
                      (5, "rgba(255,200,50,0.5)"),
                      (3, "rgba(100,255,100,0.5)")]:
    rl, rlo = make_ring(KBTP["lat"], KBTP["lon"], r_nm)
    fig.add_trace(go.Scattermap(
        lat=rl, lon=rlo, mode="lines",
        line=dict(color=colour, width=1.5),
        name=f"{r_nm} NM", hoverinfo="skip",
    ))

fig.update_layout(
    map=dict(style="carto-darkmatter",
             center=dict(lat=KBTP["lat"], lon=KBTP["lon"]), zoom=9),
    title=f"KBTP Expected Callouts — 14:00–15:00 LT  |  Hover for callout text",
    paper_bgcolor="#0A1628",
    legend=dict(bgcolor="#1A3A5C", font=dict(color="white")),
    height=650, margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()

out_html = PROJECT_ROOT / "outputs" / "kbtp_callout_map_2020-10-22.html"
fig.write_html(str(out_html))
print(f"✅ Saved → {out_html}")

✅ Saved → C:\xcas-ga-comms-assistant\outputs\kbtp_callout_map_2020-10-22.html
